# Prepare dataset for Resistance Model

In [1]:
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from axiom.config import DATA_DIR

In [2]:
TICKER = "SPY"

# SAVE_FN = f"{TICKER}_{datetime.now().strftime('%Y-%m-%d')}.csv"
SAVE_FN = "SPY_2024-07-20.csv"

SAVE_FP = DATA_DIR.joinpath("daily", SAVE_FN)

In [3]:
# # GET the data
# 
# from axiom.mdata.equity import get_daily_price_history, CandleList
# 
# data: CandleList = await get_daily_price_history(TICKER)
# 
# # Convert to DataFrame
# df = pd.DataFrame.from_records([candle.dict() for candle in data.candles])
# df.to_csv(SAVE_FP, index=False)

In [4]:
# Load the data
df = pd.read_csv(SAVE_FP)

## Process the data

In [5]:
# Filter columns
columns = ["datetime", "high", "low"]
df = df[columns]

# Convert int to datetime
df["datetime"] = pd.to_datetime(df["datetime"], unit="ms")

# Add a new column for the day of the week, 0=Monday, 6=Sunday
df["day"] = df["datetime"].dt.dayofweek

### Round the values

```
0-100 -> 0.1
100-200 -> 0.2
200-300 -> 0.3
...
```

This creates a more general resistance model that can be applied at different price levels.
It isn't perfect, but it's a good starting point.

In [6]:
# Round the high and low
def round_to_tick(value):
    tick = value // 100 or 1
    round_to = tick / 10
    return round(value / round_to) * round_to


# Apply the custom rounding to 'high' and 'low' columns
df["high"] = df["high"].apply(round_to_tick)
df["low"] = df["low"].apply(round_to_tick)

In [7]:
df.tail()

,datetime,high,low,day
5031,2024-07-15 05:00:00,565.0,559.5,0
5032,2024-07-16 05:00:00,565.0,562.0,1
5033,2024-07-17 05:00:00,560.5,556.5,2
5034,2024-07-18 05:00:00,559.5,550.5,3
5035,2024-07-19 05:00:00,554.0,548.0,4


## Prepare the dataset

Features: Last 10 trading days (highs and lows) ending on a Friday

Label: Next week's high and low prices

In [8]:
d_df = df.copy()

# Filter for Fridays (day == 4)
fridays = d_df[d_df["day"] == 4].reset_index(drop=True)

# Create features and labels for high and low prices
high_features = []
low_features = []
high_labels = []
low_labels = []

for i in range(1, len(fridays) - 1):
    friday_date = fridays.loc[i, "datetime"]
    next_friday_date = fridays.loc[i + 1, "datetime"]

    # Filter the last 10 trading days ending on this Friday
    last_5_days = d_df[
        (d_df["datetime"] <= friday_date) & (d_df["datetime"] > friday_date - pd.Timedelta(days=12))
        ]

    # Check if we have exactly 5 days
    if len(last_5_days) == 10:
        # Create the feature vector of the high and low prices of the last 5 days
        high_feature_vector = last_5_days["high"].values
        low_feature_vector = last_5_days["low"].values

        # Filter the next week's data
        next_week_data = d_df[
            (d_df["datetime"] > friday_date)
            & (d_df["datetime"] <= friday_date + pd.Timedelta(days=7))
            ]

        # Check if we have at least one day in the next week
        if not next_week_data.empty:
            # The labels are the maximum high price and minimum low price of the next week
            high_label = next_week_data["high"].max()
            low_label = next_week_data["low"].min()

            high_features.append(high_feature_vector)
            low_features.append(low_feature_vector)
            high_labels.append(high_label)
            low_labels.append(low_label)

# Convert to numpy arrays
high_X = np.array(high_features)
low_X = np.array(low_features)
high_y = np.array(high_labels)
low_y = np.array(low_labels)

# Verify the shapes
print("High Features shape:", high_X.shape)
print("Low Features shape:", low_X.shape)
print("High Labels shape:", high_y.shape)
print("Low Labels shape:", low_y.shape)

High Features shape: (695, 10)
Low Features shape: (695, 10)
High Labels shape: (695,)
Low Labels shape: (695,)


### Save the dataset

Save in JSONL format

In [15]:
import json
from pathlib import Path

data = [
    {
        "hX": high_X[i].tolist(),
        "lX": low_X[i].tolist(),
        "hy": high_y[i],
        "ly": low_y[i]
    }
    for i in range(len(high_X))
]

d_fp = Path().parent.joinpath(f"{TICKER}_resistance.jsonl")
with open(d_fp, "w") as f:
    for line in data:
        f.write(json.dumps(line) + "\n")

print(f"Dataset saved to {d_fp}")

Dataset saved to SPY_resistance.jsonl
